# Анализ данных о пассажирах Титаника
В этой работе исследуется датасет `titanic.csv`: общая информация о данных, выживаемость по классам, популярные имена, возраст пассажиров и заполненность кают.

**Файл данных:** `data/titanic.csv` (папка `data` должна находиться рядом с этим ноутбуком).

In [ ]:
# Если pandas ещё не установлен, раскомментируйте следующую строку и выполните её:
# %pip install pandas

import pandas as pd
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 1. Загрузка датасета

In [ ]:
# Путь относительно расположения этого ноутбука
df = pd.read_csv('data/titanic.csv')

print(f'Количество строк: {df.shape[0]:,}')
print(f'Количество столбцов: {df.shape[1]}')
display(df.head())

## 2. Основная информация о датасете
Посмотрим типы данных, количество непустых значений, пропуски и основные статистики.

In [ ]:
print('Информация о столбцах:')
df.info()

print('\nКоличество пропусков в каждом столбце:')
display(df.isna().sum().to_frame(name='Количество пропусков'))

print('\nДоля пропусков, %:')
display((df.isna().mean() * 100).round(2).to_frame(name='Пропуски, %'))

print('\nОписательная статистика числовых столбцов:')
display(df.describe().round(2))

print('\nОписательная статистика всех столбцов:')
display(df.describe(include='all'))

## 3. Процент выживаемости в каждом классе пассажиров
`Survived = 1` означает, что пассажир выжил, `Survived = 0` — не выжил.

In [ ]:
survival_by_class = (
    df.groupby('Pclass')['Survived']
      .mean()
      .mul(100)
      .round(2)
      .reset_index(name='Процент выживших')
)
display(survival_by_class)

## 4. Самые популярные мужское и женское имена на корабле
Из столбца `Name` извлекается первое имя после обращения (например, `Mr. Owen Harris` → `Owen`). Учитываются только строки, для которых удалось извлечь имя.

In [ ]:
def extract_first_name(full_name):
    """Извлекает первое имя после обращения в поле Name."""
    if pd.isna(full_name):
        return None
    parts = str(full_name).split(',', 1)
    if len(parts) < 2:
        return None
    after_comma = parts[1].strip()
    # Удаляем обращение (Mr., Mrs., Miss., Master. и т. п.)
    match = re.match(r'^[^ ]+\s+(.+)$', after_comma)
    if not match:
        return None
    # Берём первое слово имени; удаляем кавычки и знаки пунктуации по краям
    first_name = match.group(1).strip().split()[0]
    return re.sub(r"^[\W_]+|[\W_]+$", "", first_name)

df['FirstName'] = df['Name'].apply(extract_first_name)

def most_popular_name(data, sex_label):
    names = data.loc[data['Sex'] == sex_label, 'FirstName'].dropna()
    if names.empty:
        return 'Нет данных'
    counts = names.value_counts()
    return f'{counts.index[0]} ({counts.iloc[0]} пассажир(а))'

print('Самое популярное мужское имя:', most_popular_name(df, 'male'))
print('Самое популярное женское имя:', most_popular_name(df, 'female'))

## 5. Самые популярные мужское и женское имена в каждом классе
Для каждого значения `Pclass` отдельно определим наиболее частое имя среди мужчин и женщин.

In [ ]:
results = []
for pclass in sorted(df['Pclass'].dropna().unique()):
    class_data = df[df['Pclass'] == pclass]
    for sex, label in [('male', 'Мужское'), ('female', 'Женское')]:
        names = class_data.loc[class_data['Sex'] == sex, 'FirstName'].dropna()
        if names.empty:
            name, count = None, 0
        else:
            counts = names.value_counts()
            name, count = counts.index[0], int(counts.iloc[0])
        results.append({
            'Класс пассажира': int(pclass),
            'Тип имени': label,
            'Самое популярное имя': name,
            'Количество': count
        })

display(pd.DataFrame(results))

## 6. Пассажиры старше 44 лет

In [ ]:
passengers_over_44 = df[df['Age'] > 44]
print(f'Количество пассажиров старше 44 лет: {len(passengers_over_44)}')
display(passengers_over_44.head(10))

## 7. Мужчины младше 44 лет

In [ ]:
men_under_44 = df[(df['Age'] < 44) & (df['Sex'] == 'male')]
print(f'Количество мужчин младше 44 лет: {len(men_under_44)}')
display(men_under_44.head(10))

## 8. Количество кают по числу проживавших в них пассажиров
В `Cabin` иногда записано несколько номеров кают через пробел. В этой работе считаем одинаковую запись в ячейке одной группой кают и подсчитываем, сколько пассажиров имеют такую же запись. Пустые значения не учитываются.

In [ ]:
cabin_counts = df['Cabin'].dropna().value_counts()
occupancy_counts = cabin_counts.value_counts().sort_index()
cabin_occupancy = occupancy_counts.rename_axis('Число пассажиров в записи каюты').reset_index(name='Количество таких записей')
display(cabin_occupancy)

print('Например, строка с числом 2 показывает, сколько записей кают встречается у двух пассажиров.')

## Вывод
По результатам выполнения ячеек можно сравнить выживаемость пассажиров разных классов, найти наиболее распространённые имена, изучить возрастные группы и распределение пассажиров по каютам.